In [ ]:
import os
import re
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW  

from transformers import AutoTokenizer, AutoModel
try:
    from transformers import get_linear_schedule_with_warmup
except ImportError:
    from transformers.optimization import get_linear_schedule_with_warmup

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score


In [ ]:
# =====================
# CONFIG
# =====================
TRAIN_PATH = "/kaggle/input/competitions/go-data-science-5-0/train.csv"
TEST_PATH = "/kaggle/input/competitions/go-data-science-5-0/test.csv"
SAMPLE_SUB_PATH = "/kaggle/input/competitions/go-data-science-5-0/sample_submission.csv"   # optional
OUT_PATH = "/kaggle/working/submission.csv"

OUT_PROBS = "submission_probs.csv"
OUT_BIN = "submission_binary.csv"

ID_COL = "id"
TEXT_COL = "text"
LABEL_COLS = ["E", "S", "G", "non_ESG"]

MODEL_NAME = "microsoft/deberta-v3-base"
MAX_LEN = 256
BATCH_SIZE = 4
LR = 2e-5
EPOCHS = 2
N_SPLITS = 5
SEED = 42
NUM_WORKERS = min(4, os.cpu_count() or 2)
THRESH = 0.5

# Print control
PRINT_EVERY = 200          # batches
PRINT_FIRST_N_BATCHES = 3  # always print first few batches in each epoch

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [ ]:
class ESGDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts = texts
        self.labels = labels  # None for test/val-prob pass
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            truncation=True,
            max_length=self.max_len,
            padding="max_length",
            return_tensors="pt",
        )
        item = {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
        }
        if self.labels is not None:
            item["labels"] = torch.tensor(self.labels[idx], dtype=torch.float32)
        return item

class MultiLabelClassifier(nn.Module):
    def __init__(self, model_name: str, num_labels: int):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(model_name)
        hidden = self.backbone.config.hidden_size
        self.dropout = nn.Dropout(0.2)
        self.head = nn.Linear(hidden, num_labels)
        nn.init.xavier_normal_(self.head.weight)

    def forward(self, input_ids, attention_mask):
        out = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
        cls = out.last_hidden_state[:, 0, :]
        logits = self.head(self.dropout(cls))
        return logits

def compute_pos_weight(y: np.ndarray) -> torch.Tensor:
    y = y.astype(np.float32)
    pos = y.sum(axis=0)
    neg = y.shape[0] - pos
    pw = (neg / np.clip(pos, 1.0, None)).astype(np.float32)
    return torch.tensor(pw, device=DEVICE)


In [ ]:
# =====================
# LOGGING HELPERS
# =====================
def log(msg: str):
    ts = time.strftime("%H:%M:%S")
    print(f"[{ts}] {msg}", flush=True)

def cuda_mem():
    if not torch.cuda.is_available():
        return ""
    alloc = torch.cuda.memory_allocated() / (1024**3)
    reserved = torch.cuda.memory_reserved() / (1024**3)
    return f" | cuda_mem alloc={alloc:.2f}GB reserved={reserved:.2f}GB"


# =====================
# HELPERS
# =====================
def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def basic_clean(s: str) -> str:
    s = str(s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def make_stratify_key(y_df: pd.DataFrame, n_splits: int) -> np.ndarray:
    combos = y_df.astype(int).astype(str).agg("".join, axis=1)
    vc = combos.value_counts()
    rare = set(vc[vc < n_splits].index)
    combos = combos.where(~combos.isin(rare), "__RARE__")
    return combos.values

@torch.no_grad()
def predict_probs(model, loader, desc="predict"):
    model.eval()
    all_probs = []
    t0 = time.time()
    n_batches = len(loader)
    for i, batch in enumerate(loader, 1):
        input_ids = batch["input_ids"].to(DEVICE)
        attn = batch["attention_mask"].to(DEVICE)
        logits = model(input_ids=input_ids, attention_mask=attn)
        probs = torch.sigmoid(logits).detach().cpu().numpy()
        all_probs.append(probs)

        if i == 1 or i == n_batches or (i % max(1, n_batches // 5) == 0):
            log(f"{desc}: batch {i}/{n_batches}{cuda_mem()}")

    log(f"{desc}: done in {time.time()-t0:.1f}s")
    return np.vstack(all_probs)

def train_one_fold(fold, train_idx, val_idx, df, tokenizer, pos_weight):
    log(f"=== Fold {fold} start ===")
    log(f"train_idx={len(train_idx)} val_idx={len(val_idx)}")

    train_texts = df.loc[train_idx, TEXT_COL].tolist()
    val_texts = df.loc[val_idx, TEXT_COL].tolist()

    y_train = df.loc[train_idx, LABEL_COLS].values.astype(np.float32)
    y_val = df.loc[val_idx, LABEL_COLS].values.astype(np.float32)

    log(f"y_train positives per label: {dict(zip(LABEL_COLS, y_train.sum(axis=0).astype(int).tolist()))}")
    log(f"y_val   positives per label: {dict(zip(LABEL_COLS, y_val.sum(axis=0).astype(int).tolist()))}")
    log(f"pos_weight: {dict(zip(LABEL_COLS, pos_weight.detach().cpu().numpy().round(2).tolist()))}")

    train_ds = ESGDataset(train_texts, y_train, tokenizer, MAX_LEN)
    val_ds = ESGDataset(val_texts, y_val, tokenizer, MAX_LEN)

    train_loader = DataLoader(
        train_ds,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
    )
    val_loader = DataLoader(
        val_ds,
        batch_size=BATCH_SIZE * 2,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
    )

    log(f"train_loader batches={len(train_loader)} val_loader batches={len(val_loader)}")

    model = MultiLabelClassifier(MODEL_NAME, num_labels=len(LABEL_COLS)).to(DEVICE)
    log(f"Model loaded on {DEVICE}{cuda_mem()}")

    optimizer = AdamW(model.parameters(), lr=LR)
    total_steps = len(train_loader) * EPOCHS
    scheduler = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=0, num_training_steps=total_steps
    )
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    best_f1 = -1.0
    best_state = None

    for epoch in range(1, EPOCHS + 1):
        log(f"--- Fold {fold} | Epoch {epoch}/{EPOCHS} ---")
        model.train()
        t0 = time.time()

        running_loss = 0.0
        n_batches = len(train_loader)

        for bi, batch in enumerate(train_loader, 1):
            input_ids = batch["input_ids"].to(DEVICE)
            attn = batch["attention_mask"].to(DEVICE)
            labels = batch["labels"].to(DEVICE)

            logits = model(input_ids=input_ids, attention_mask=attn)
            loss = criterion(logits, labels)

            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            optimizer.step()
            scheduler.step()

            running_loss += loss.item()

            # Prints
            if bi <= PRINT_FIRST_N_BATCHES or bi == n_batches or (bi % PRINT_EVERY == 0):
                avg_loss = running_loss / bi
                log(f"fold {fold} epoch {epoch} batch {bi}/{n_batches} "
                    f"loss={loss.item():.4f} avg_loss={avg_loss:.4f}{cuda_mem()}")

        train_avg_loss = running_loss / max(1, n_batches)
        log(f"Epoch train done in {time.time()-t0:.1f}s | avg_loss={train_avg_loss:.4f}{cuda_mem()}")

        # Validate
        t1 = time.time()
        val_probs = predict_probs(model, val_loader, desc=f"fold {fold} val")
        val_pred = (val_probs >= THRESH).astype(int)
        f1 = f1_score(y_val, val_pred, average="macro", zero_division=0)
        log(f"Validation done in {time.time()-t1:.1f}s | val_macro_f1={f1:.4f}")

        # Optional: show per-label predicted positive rate
        pred_pos_rate = val_pred.mean(axis=0).round(3)
        log(f"Pred pos-rate per label: {dict(zip(LABEL_COLS, pred_pos_rate.tolist()))}")

        if f1 > best_f1:
            best_f1 = f1
            best_state = {k: v.detach().cpu() for k, v in model.state_dict().items()}
            log(f"New best model for fold {fold}! best_f1={best_f1:.4f}")

    if best_state is not None:
        model.load_state_dict(best_state, strict=True)
        log(f"Fold {fold} loaded best checkpoint best_f1={best_f1:.4f}")

    log(f"=== Fold {fold} end ===")
    return model, best_f1


# =====================
# MAIN
# =====================
def main():
    seed_everything(SEED)
    log(f"Starting run | device={DEVICE} | torch={torch.__version__}")
    log(f"Config: model={MODEL_NAME} max_len={MAX_LEN} bs={BATCH_SIZE} lr={LR} epochs={EPOCHS} folds={N_SPLITS}")
    train_path = TRAIN_PATH if Path(TRAIN_PATH).exists() else ALT_TRAIN_PATH
    if not Path(train_path).exists():
        raise FileNotFoundError(f"Could not find {TRAIN_PATH} or {ALT_TRAIN_PATH}")

    if not Path(TEST_PATH).exists():
        raise FileNotFoundError(f"Could not find {TEST_PATH} (put your test file in the same folder or edit TEST_PATH)")

    log(f"Reading train from: {train_path}")
    train_df = pd.read_csv(train_path)
    log(f"Reading test from:  {TEST_PATH}")
    test_df = pd.read_csv(TEST_PATH)

    log(f"Train shape={train_df.shape} | Test shape={test_df.shape}")
    log(f"Train columns: {list(train_df.columns)}")
    log(f"Test columns:  {list(test_df.columns)}")

    # Validate columns
    need_train = [ID_COL, TEXT_COL] + LABEL_COLS
    missing_train = [c for c in need_train if c not in train_df.columns]
    if missing_train:
        raise ValueError(f"Train missing columns: {missing_train}")

    need_test = [ID_COL, TEXT_COL]
    missing_test = [c for c in need_test if c not in test_df.columns]
    if missing_test:
        raise ValueError(f"Test missing columns: {missing_test}")

    # Clean text
    log("Cleaning text...")
    train_df[TEXT_COL] = train_df[TEXT_COL].fillna("").map(basic_clean)
    test_df[TEXT_COL] = test_df[TEXT_COL].fillna("").map(basic_clean)
    log(f"Example train text: {train_df[TEXT_COL].iloc[0][:200]}")

    # Label stats
    y_all = train_df[LABEL_COLS].values.astype(np.float32)
    pos_counts = y_all.sum(axis=0).astype(int)
    pos_rate = (pos_counts / len(train_df)).round(3)
    log(f"Label positive counts: {dict(zip(LABEL_COLS, pos_counts.tolist()))}")
    log(f"Label positive rates:  {dict(zip(LABEL_COLS, pos_rate.tolist()))}")

    log("Loading tokenizer...")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    log("Tokenizer loaded.")

    pos_weight = compute_pos_weight(y_all)
    log(f"pos_weight: {dict(zip(LABEL_COLS, pos_weight.detach().cpu().numpy().round(2).tolist()))}")

    strat_key = make_stratify_key(train_df[LABEL_COLS], N_SPLITS)
    log(f"Stratify combos unique={pd.Series(strat_key).nunique()} (rare bucketed)")

    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

    oof = np.zeros((len(train_df), len(LABEL_COLS)), dtype=np.float32)
    test_probs = np.zeros((len(test_df), len(LABEL_COLS)), dtype=np.float32)

    # Test loader once
    log("Building test DataLoader...")
    test_ds = ESGDataset(test_df[TEXT_COL].tolist(), labels=None, tokenizer=tokenizer, max_len=MAX_LEN)
    test_loader = DataLoader(
        test_ds,
        batch_size=BATCH_SIZE * 2,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
    )
    log(f"Test loader batches={len(test_loader)}")

    fold_scores = []

    for fold, (tr_idx, va_idx) in enumerate(skf.split(train_df, strat_key), 1):
        model, best_f1 = train_one_fold(fold, tr_idx, va_idx, train_df, tokenizer, pos_weight)
        fold_scores.append(best_f1)

        log(f"Predicting OOF for fold {fold}...")
        val_texts = train_df.loc[va_idx, TEXT_COL].tolist()
        val_ds = ESGDataset(val_texts, labels=None, tokenizer=tokenizer, max_len=MAX_LEN)
        val_loader = DataLoader(
            val_ds,
            batch_size=BATCH_SIZE * 2,
            shuffle=False,
            num_workers=NUM_WORKERS,
            pin_memory=torch.cuda.is_available(),
        )
        oof[va_idx] = predict_probs(model, val_loader, desc=f"fold {fold} oof")

        log(f"Predicting test for fold {fold} (averaging)...")
        test_probs += predict_probs(model, test_loader, desc=f"fold {fold} test") / N_SPLITS

        del model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        log(f"Fold {fold} cleanup done.{cuda_mem()}")

    log(f"Fold macro-F1: {[round(x, 4) for x in fold_scores]} | mean={round(float(np.mean(fold_scores)), 4)}")

    # Save probability submission
    log("Writing submissions...")
    sub_probs = pd.DataFrame({ID_COL: test_df[ID_COL].values})
    for j, c in enumerate(LABEL_COLS):
        sub_probs[c] = test_probs[:, j]

    if Path(SAMPLE_SUB_PATH).exists():
        log(f"Found sample submission: {SAMPLE_SUB_PATH} (aligning columns)")
        sample = pd.read_csv(SAMPLE_SUB_PATH)
        keep = [c for c in sample.columns if c in sub_probs.columns]
        sub_probs = sub_probs[keep]
        log(f"Submission columns after align: {list(sub_probs.columns)}")

    sub_probs.to_csv(OUT_PROBS, index=False)
    log(f"Saved {OUT_PROBS} shape={sub_probs.shape}")

    # Binary submission
    sub_bin = sub_probs.copy()
    for c in LABEL_COLS:
        if c in sub_bin.columns:
            sub_bin[c] = (sub_bin[c].astype(float) >= THRESH).astype(int)

    sub_bin.to_csv(OUT_BIN, index=False)
    log(f"Saved {OUT_BIN} shape={sub_bin.shape}")


if __name__ == "__main__":
    main()


[22:19:00] Starting run | device=cuda | torch=2.8.0+cu126
[22:19:00] Config: model=microsoft/deberta-v3-base max_len=256 bs=4 lr=2e-05 epochs=2 folds=5
[22:19:00] Reading train from: /kaggle/input/competitions/go-data-science-5-0/train.csv
[22:19:00] Reading test from:  /kaggle/input/competitions/go-data-science-5-0/test.csv
[22:19:00] Train shape=(26750, 6) | Test shape=(2000, 2)
[22:19:00] Train columns: ['id', 'text', 'E', 'S', 'G', 'non_ESG']
[22:19:00] Test columns:  ['id', 'text']
[22:19:00] Cleaning text...
[22:19:00] Example train text: In addition to shares held by the Family Trust, members of the Slim Family, including Carlos Slim Helú, directly own an aggregate of of each series.
[22:19:00] Label positive counts: {'E': 1180, 'S': 9910, 'G': 8857, 'non_ESG': 11389}
[22:19:00] Label positive rates:  {'E': 0.044, 'S': 0.37, 'G': 0.331, 'non_ESG': 0.426}
[22:19:00] Loading tokenizer...


/usr/local/lib/python3.12/dist-packages/transformers/convert_slow_tokenizer.py:564: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


[22:19:01] Tokenizer loaded.
[22:19:01] pos_weight: {'E': 21.670000076293945, 'S': 1.7000000476837158, 'G': 2.0199999809265137, 'non_ESG': 1.350000023841858}
[22:19:01] Stratify combos unique=10 (rare bucketed)
[22:19:01] Building test DataLoader...
[22:19:01] Test loader batches=250
[22:19:01] === Fold 1 start ===
[22:19:01] train_idx=21400 val_idx=5350
[22:19:01] y_train positives per label: {'E': 944, 'S': 7928, 'G': 7086, 'non_ESG': 9111}
[22:19:01] y_val   positives per label: {'E': 236, 'S': 1982, 'G': 1771, 'non_ESG': 2278}
[22:19:01] pos_weight: {'E': 21.670000076293945, 'S': 1.7000000476837158, 'G': 2.0199999809265137, 'non_ESG': 1.350000023841858}
[22:19:01] train_loader batches=5350 val_loader batches=669


2026-02-14 22:19:02.571157: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1771107542.592382     536 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1771107542.599405     536 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1771107542.616697     536 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1771107542.616717     536 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1771107542.616720     536 computation_placer.cc:177] computation placer alr

[22:19:06] Model loaded on cuda | cuda_mem alloc=0.69GB reserved=0.72GB
[22:19:06] --- Fold 1 | Epoch 1/2 ---
[22:19:07] fold 1 epoch 1 batch 1/5350 loss=0.9461 avg_loss=0.9461 | cuda_mem alloc=2.81GB reserved=4.01GB
[22:19:07] fold 1 epoch 1 batch 2/5350 loss=1.0027 avg_loss=0.9744 | cuda_mem alloc=2.81GB reserved=4.94GB
[22:19:08] fold 1 epoch 1 batch 3/5350 loss=1.4261 avg_loss=1.1250 | cuda_mem alloc=2.81GB reserved=4.94GB
[22:20:09] fold 1 epoch 1 batch 200/5350 loss=1.4308 avg_loss=1.0373 | cuda_mem alloc=2.81GB reserved=4.94GB
[22:21:11] fold 1 epoch 1 batch 400/5350 loss=0.9550 avg_loss=1.0599 | cuda_mem alloc=2.80GB reserved=4.94GB
[22:22:12] fold 1 epoch 1 batch 600/5350 loss=1.0250 avg_loss=1.0536 | cuda_mem alloc=2.80GB reserved=4.94GB
[22:23:14] fold 1 epoch 1 batch 800/5350 loss=2.0886 avg_loss=1.0325 | cuda_mem alloc=2.81GB reserved=4.94GB
[22:24:15] fold 1 epoch 1 batch 1000/5350 loss=0.7470 avg_loss=1.0291 | cuda_mem alloc=2.81GB reserved=4.94GB
[22:25:17] fold 1 epoch